**Learning Objectives:**
1. Learn how to use Agent Platform AutoML pre-built components
1. Learn how to build the Agent Platform AutoML pipeline with these components using BigQuery as a data source
1. Learn how to compile, upload, and run the Agent Platform AutoML pipeline


In this lab, you will build, deploy, and run the Agent Platform AutoML pipeline that orchestrates the **Agent Platform AutoML AI** services to train, tune, and serve batch predictions to BigQuery with a model. 

## Setup

In [ ]:
import os

from google.cloud import aiplatform

%load_ext google.cloud.bigquery

In [ ]:
REGION = "us-central1"
PROJECT = !(gcloud config get-value project)
PROJECT = PROJECT[0]

os.environ["PROJECT"] = PROJECT

### BigQuery Data

If you have not gone through the KFP Walkthrough lab, you will need to run the following cell to create a BigQuery dataset and table containing the data required for this lab.

In [ ]:
%%bash
DATASET_LOCATION=US
DATASET_ID=power_grid
TABLE_ID=grid_load_train_demo
DATA_SOURCE=gs://asl-public-data/data/power_grid_load/grid_load_demo_dataset.csv
SCHEMA=timestamp:TIMESTAMP,\
line_id:STRING,\
feeder_id:STRING,\
substation_id:STRING,\
grid_level:STRING,\
substation_type:STRING,\
load_mw:FLOAT64,\
design_capacity_mw:FLOAT64,\
temperature_c:FLOAT64,\
is_weekend:STRING

bq --location=$DATASET_LOCATION --project_id=$PROJECT mk --dataset $DATASET_ID

bq --project_id=$PROJECT --dataset_id=$DATASET_ID load \
--source_format=CSV \
--skip_leading_rows=1 \
--replace \
$TABLE_ID \
$DATA_SOURCE \
$SCHEMA

## Understanding the pipeline design


The workflow implemented by the pipeline is defined using a Python based Domain Specific Language (DSL). The pipeline's DSL is in the `pipeline/pipeline_automl_batch_preds.py` file that we will generate below.

The pipeline's DSL has been designed to avoid hardcoding any environment specific settings like file paths or connection strings. These settings are provided to the pipeline code through a set of environment variables.


## Building and deploying the pipeline

Let us write the pipeline to disk:

In [ ]:
%%writefile ./pipeline/power_grid_pipeline_automl.py
"""Kubeflow Power-Grid Pipeline."""

import os

from google_cloud_pipeline_components.v1.automl.training_job import (
    AutoMLTabularTrainingJobRunOp, AutoMLForecastingTrainingJobRunOp
)

from google_cloud_pipeline_components.v1.dataset import ( 
    TabularDatasetCreateOp, TimeSeriesDatasetCreateOp
)

from google_cloud_pipeline_components.v1.batch_predict_job import (
    ModelBatchPredictOp
)

from kfp import dsl

PIPELINE_ROOT = os.getenv("PIPELINE_ROOT")
PROJECT = os.getenv("PROJECT")
REGION = os.getenv("REGION")
DATASET_SOURCE = os.getenv("DATASET_SOURCE")
PIPELINE_NAME = os.getenv("PIPELINE_NAME", "covertype")
DISPLAY_NAME = os.getenv("MODEL_DISPLAY_NAME", PIPELINE_NAME)
TARGET_COLUMN = os.getenv("TARGET_COLUMN", "Cover_Type")
BATCH_PREDS_SOURCE_URI = os.getenv("BATCH_PREDS_SOURCE_URI")
# Agent Platform AutoML Component image version
KFP_IMAGE = "gcr.io/ml-pipeline/google-cloud-pipeline-components:2.16.1"

@dsl.pipeline(
    name=f"{PIPELINE_NAME}-automl-pipeline",
    description=f"AutoML Pipeline for {PIPELINE_NAME}",
    pipeline_root=PIPELINE_ROOT,
)
def create_pipeline():

    bq_table_uri=f"bq://{PROJECT}.power_grid.grid_load_train_demo"

    # Register the BigQuery table as a Vertex AI Managed TimeSeriesDataset
    dataset_create_task = TimeSeriesDatasetCreateOp(
        project=PROJECT,
        location=REGION,
        display_name="dataset_grid_load_train_demo",
        bq_source=bq_table_uri,
    )

    # 1. Define Column specs matching your CSV dtypes
    column_specs = {
        "timestamp": "timestamp",
        "load_mw": "numeric",
        "temperature_c": "numeric",
        "is_weekend": "numeric",
        "design_capacity_mw": "numeric",
        "grid_level": "categorical",
        "substation_type": "categorical",
        "feeder_id": "categorical",
        "substation_id": "categorical"
    }

    # 2. Configure and run the Forecasting Model Training Job
    automl_training_task = AutoMLForecastingTrainingJobRunOp(
        project=PROJECT,
        location=REGION,
        display_name="power_grid_forecasting",
        # Pass the dataset registered in step 1
        dataset=dataset_create_task.outputs["dataset"],

        # Define target, timestamp, and identifier columns
        target_column="load_mw",
        time_column="timestamp",  # <-- Sets the timestamp column
        time_series_identifier_column="line_id", # <-- Sets the multi-time-series ID columns

        # Additional forecasting configurations
        data_granularity_unit="hour",      # <-- Set granularity to hourly
        data_granularity_count=1,          # <-- 1-hour increments
        forecast_horizon=24,               # <-- Forecast next 24 hours
        context_window=168,                # <-- Look back 168 hours (7 days of history)

        # --- Hierarchy Configuration (Direct Top-Level Arguments) ---
        hierarchy_group_columns=["substation_id", "feeder_id"],
        hierarchy_group_total_weight=1.0,
        hierarchy_temporal_total_weight=0.5,
        hierarchy_group_temporal_total_weight=1.0,

        # --- Transformations & Features ---
        column_specs=column_specs,
        available_at_forecast_columns=["temperature_c", "is_weekend","timestamp"],
        unavailable_at_forecast_columns = ["load_mw"],

        # Constant attributes:
        time_series_attribute_columns=[
            "design_capacity_mw",
            "grid_level",
            "substation_type",
            "feeder_id",
            "substation_id",
        ],

        # Optimization Metric
        optimization_objective="minimize-rmse",
        
        budget_milli_node_hours=1000,
    )

Overwriting ./pipeline/pipeline_automl_batch_preds.py

### Compile the pipeline

Let's start by defining the environment variables that will be passed to the pipeline compiler:

In [ ]:
ARTIFACT_STORE = f"gs://{PROJECT}-kfp-artifact-store"
PIPELINE_ROOT = f"{ARTIFACT_STORE}/pipeline"
DATASET_SOURCE = f"bq://{PROJECT}.power_grid.grid_load_demo"
BATCH_PREDS_SOURCE_URI = f"bq://{PROJECT}.power_grid.grid_load_forecast_demo"

%env PIPELINE_ROOT={PIPELINE_ROOT}
%env PROJECT={PROJECT}
%env REGION={REGION}
%env DATASET_SOURCE={DATASET_SOURCE}
%env BATCH_PREDS_SOURCE_URI={BATCH_PREDS_SOURCE_URI}

Let us make sure that the `ARTIFACT_STORE` has been created, and let us create it if not:

In [ ]:
!gcloud storage ls | grep ^{ARTIFACT_STORE}/$ || gcloud storage buckets create --location={REGION} {ARTIFACT_STORE}

#### Use the CLI compiler to compile the pipeline

We compile the pipeline from the Python file we generated into a YAML description using the following command:

In [ ]:
PIPELINE_YAML = "power_grid_pipeline_automl.yaml"

In [ ]:
!kfp dsl compile --py pipeline/power_grid_pipeline_automl.py --output $PIPELINE_YAML

/home/jupyter/asl-ml-immersion/notebooks/kubeflow_pipelines/pipelines/solutions/covertype_automl_pipeline_batch_preds.yaml

**Note:** You can also use the Python SDK to compile the pipeline:

```python
from kfp import compiler

compiler.Compiler().compile(
    pipeline_func=create_pipeline, 
    package_path=PIPELINE_YAML,
)

```

The result is the pipeline file. 

In [ ]:
!head {PIPELINE_YAML}

### Deploy the pipeline package

In [ ]:
aiplatform.init(project=PROJECT, location=REGION)

pipeline = aiplatform.PipelineJob(
    display_name="automl_power_grid_kfp_train_pipeline",
    template_path=PIPELINE_YAML,
    enable_caching=True,
)

pipeline.run()

Copyright 2026 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

    https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.